In [ ]:
import os

# Send diagnostic plots to a temporary folder; the paper renderer handles final assets.
from pathlib import Path as _NotebookPath
import tempfile as _notebook_tempfile
NOTEBOOK_FIGURE_DIR = _NotebookPath(
    os.environ.get("ANNALS_NOTEBOOK_FIGURE_DIR")
    or _notebook_tempfile.mkdtemp(prefix="annals-pareto_family_correlated-")
)
NOTEBOOK_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm, pareto

# Avoid Matplotlib cache warnings on some setups
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "axes.grid": True,
})

rng = np.random.default_rng(0)

# Default correlation used for 1D / 2D visualizations (cells 2-3).
# Cell 5 sweeps multiple rho values to demonstrate invariance.
rho_default = 0.5


In [ ]:
def correlation_cholesky(rho: float) -> np.ndarray:
    """Lower-triangular Cholesky factor L of the 2x2 correlation matrix
    [[1, rho], [rho, 1]] (unit diagonal). Then L @ L.T = [[1, rho], [rho, 1]].
    """
    if not -1.0 < rho < 1.0:
        raise ValueError(f"rho must lie in (-1, 1); got {rho}")
    return np.array([[1.0, 0.0],
                     [rho, np.sqrt(1.0 - rho ** 2)]])


def gaussian_to_pareto_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Marginal 1D pushforward N(0,1) -> Pareto(alpha).

    Pareto Type I, in SciPy: pareto(b=alpha), scale = 1.
    """
    u = norm.cdf(x)
    u = np.clip(u, eps, 1.0 - eps)
    return pareto.ppf(u, b=alpha)


def jacobian_diag_entry_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    y = gaussian_to_pareto_1d(x, alpha=alpha, eps=eps)
    return norm.pdf(x) / pareto.pdf(y, b=alpha)


def transport_2d_correlated(X: np.ndarray, alpha: float, rho: float, eps: float = 1e-12) -> np.ndarray:
    """T(x) = L @ (q(x_1), q(x_2))^T where q is the marginal quantile map.

    Cov(T(X)) = sigma_marginal^2 * [[1, rho], [rho, 1]] (correlated target with
    same marginal distribution as the iid case).
    """
    X = np.asarray(X)
    assert X.shape[-1] == 2
    Z1 = gaussian_to_pareto_1d(X[..., 0], alpha=alpha, eps=eps)
    Z2 = gaussian_to_pareto_1d(X[..., 1], alpha=alpha, eps=eps)
    L = correlation_cholesky(rho)
    Y = np.empty_like(X, dtype=float)
    Y[..., 0] = L[0, 0] * Z1 + L[0, 1] * Z2  # = Z1
    Y[..., 1] = L[1, 0] * Z1 + L[1, 1] * Z2  # = rho Z1 + sqrt(1-rho^2) Z2
    return Y


def jacobian_2d_correlated(X: np.ndarray, alpha: float, rho: float, eps: float = 1e-12) -> np.ndarray:
    """Full 2x2 Jacobian J_T(x) = L @ diag(j(x_1), j(x_2))."""
    X = np.asarray(X)
    assert X.shape[-1] == 2
    j1 = jacobian_diag_entry_1d(X[..., 0], alpha=alpha, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], alpha=alpha, eps=eps)
    L = correlation_cholesky(rho)
    J = np.zeros(X.shape[:-1] + (2, 2), dtype=float)
    J[..., 0, 0] = L[0, 0] * j1
    J[..., 0, 1] = L[0, 1] * j2
    J[..., 1, 0] = L[1, 0] * j1
    J[..., 1, 1] = L[1, 1] * j2
    return J


def log_abs_det_jacobian_2d_correlated(X: np.ndarray, alpha: float, rho: float, eps: float = 1e-12) -> np.ndarray:
    """log|det J_T(x)| = log|det L| + log|j(x_1)| + log|j(x_2)|.

    det L = sqrt(1 - rho^2) > 0 for rho in (-1, 1).
    """
    X = np.asarray(X)
    j1 = jacobian_diag_entry_1d(X[..., 0], alpha=alpha, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], alpha=alpha, eps=eps)
    log_det_L = 0.5 * np.log1p(-rho ** 2)
    return log_det_L + np.log(np.abs(j1)) + np.log(np.abs(j2))


In [ ]:
# --- Plots: marginal Jacobian j_{alpha}(x) of the coordinate quantile map ---
# (The full transport is T(x) = L @ diag(j(x_1), j(x_2)); the marginal entries
#  j(x) below are the diagonal of D before A-multiplication.)

alphas = [2.1, 3.0, 5.0, 10.0, 30.0]

x = np.linspace(-4.0, 4.0, 2001)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)

ax = axes[0]
for a in alphas:
    j = jacobian_diag_entry_1d(x, alpha=a)
    ax.plot(x, j, label=f"alpha={a}")
ax.set_title(r"Marginal Jacobian $j_\alpha(x)$")
ax.set_xlabel("x")
ax.set_ylabel(r"$j_\alpha(x)$")
ax.set_yscale("log")
ax.legend(ncol=2, fontsize=8)

ax = axes[1]
for a in alphas:
    j = jacobian_diag_entry_1d(x, alpha=a)
    ax.plot(x, np.log(j), label=f"alpha={a}")
ax.set_title(r"$\log j_\alpha(x)$")
ax.set_xlabel("x")
ax.set_ylabel(r"$\log j_\alpha(x)$")
ax.legend(ncol=2, fontsize=8)

plt.show()


In [ ]:
# --- 2D heatmaps of log|det J_T(x)| for the correlated transport (rho = rho_default) ---

grid_lim = 4.0
n_grid = 301
x1 = np.linspace(-grid_lim, grid_lim, n_grid)
x2 = np.linspace(-grid_lim, grid_lim, n_grid)
X1, X2 = np.meshgrid(x1, x2, indexing="xy")
Xgrid = np.stack([X1, X2], axis=-1)  # (n_grid, n_grid, 2)

fig, axes = plt.subplots(1, len(alphas), figsize=(3.2 * len(alphas), 3.1), constrained_layout=True)
if len(alphas) == 1:
    axes = [axes]

# precompute global color limits for comparability
vals = []
for a in alphas:
    vals.append(log_abs_det_jacobian_2d_correlated(Xgrid, alpha=a, rho=rho_default))
allv = np.concatenate([v.ravel() for v in vals])
vmin, vmax = np.quantile(allv, [0.02, 0.98])

for ax, a, Z in zip(axes, alphas, vals):
    im = ax.imshow(
        Z,
        origin="lower",
        extent=[-grid_lim, grid_lim, -grid_lim, grid_lim],
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="equal",
    )
    ax.set_title(rf"$\alpha={a}$", fontsize=16)

cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label(rf"$\log|\det J_T|$  ($\rho={rho_default}$)", fontsize=14)

plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "logdet_heatmaps_pareto_correlated.png", dpi=150, bbox_inches="tight")


In [ ]:
from numpy.polynomial.hermite import hermgauss


def log_jacobian_diag_entry_1d(x: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """Stable log of j_alpha(x)."""
    u = norm.cdf(x)
    u = np.clip(u, eps, 1.0 - eps)
    y = pareto.ppf(u, b=alpha)
    return norm.logpdf(x) - pareto.logpdf(y, b=alpha)


def normal_expectation_gauss_hermite_1d(fun, n: int = 80) -> float:
    xs, ws = hermgauss(n)
    z = np.sqrt(2.0) * xs
    vals = fun(z)
    return float(np.sum(ws * vals) / np.sqrt(np.pi))


def estimate_E_jsq(  alpha: float, n_gh: int = 80) -> float:
    """E_{Z~N(0,1)}[ j(Z)^2 ] for the marginal map."""
    def j2(z):
        logj = log_jacobian_diag_entry_1d(z, alpha=alpha)
        return np.exp(np.clip(2.0 * logj, -745.0, 709.0))
    return normal_expectation_gauss_hermite_1d(j2, n=n_gh)


def estimate_E_frob_sq_jac_2d_correlated(alpha: float, rho: float, n_gh: int = 80) -> float:
    """E_{X~N(0,I_2)}[ ||J_T(X)||_F^2 ] for T(x) = L @ diag(j(x_1), j(x_2)).

    ||J_T||_F^2 = (1 + rho^2) j(x_1)^2 + (1 - rho^2) j(x_2)^2,
    so E[||J_T||_F^2] = ((1+rho^2) + (1-rho^2)) * E[j^2] = 2 * E[j^2].
    Independent of rho — the formula below is written generally so that the
    invariance is *checked numerically* in the comparison cell.
    """
    Ej2 = estimate_E_jsq(alpha, n_gh=n_gh)
    L = correlation_cholesky(rho)
    col_norm_sq_0 = L[0, 0] ** 2 + L[1, 0] ** 2     # = 1 + rho^2
    col_norm_sq_1 = L[0, 1] ** 2 + L[1, 1] ** 2     # = 1 - rho^2
    return col_norm_sq_0 * Ej2 + col_norm_sq_1 * Ej2


In [ ]:
# --- Poincaré comparison: Jacobian Size for several rho's vs Scaled Lower Bound ---
# Headline: with target Y = L @ Z (correlation-matrix factor with unit diagonal),
# E[||J_T||_F^2] is independent of rho, so all curves overlap.

alpha_grid = np.concatenate([
    np.linspace(2.05, 5.0, 30),
    np.linspace(5.0, 20.0, 31)[1:],
    np.linspace(20.0, 80.0, 25)[1:],
])

n_gh = 120
rhos_to_plot = [0.0, 0.5, 0.9]

rhs_per_rho = {
    r: np.array([estimate_E_frob_sq_jac_2d_correlated(a, rho=r, n_gh=n_gh)
                 for a in alpha_grid])
    for r in rhos_to_plot
}

# "LHS" overlay: trace of Cov(Y) = sigma_marginal^2 * trace([[1,rho],[rho,1]]) = 2 * Var(target).
# The trace is unchanged by correlation, so the lower bound is the same as in the iid case.
# Pareto Type I (x_m = 1):
#   Var = alpha / ((alpha - 1)^2 (alpha - 2))   for alpha > 2.
lhs_var_target = 2.0 * (alpha_grid / ((alpha_grid - 1.0) ** 2 * (alpha_grid - 2.0)))


# Same convention as the previous notebooks: divide the lower bound by the constant.
C_poincare = 1.0
scaled_lower_bound = lhs_var_target / C_poincare

# ---- numerical invariance check ----
ref = rhs_per_rho[rhos_to_plot[0]]
max_relerr = 0.0
for r in rhos_to_plot:
    diff = np.abs(rhs_per_rho[r] - ref) / np.maximum(ref, 1e-300)
    max_relerr = max(max_relerr, float(np.max(diff)))
print(f"max relative error across rho in {rhos_to_plot}: {max_relerr:.2e}  (expect ~quadrature noise)")

# ---- plot ----
colors = {0.0: "#1f77b4", 0.5: "#9467bd", 0.9: "#2ca02c"}
markers = {0.0: "o", 0.5: "s", 0.9: "^"}

fig, ax = plt.subplots(1, 1, figsize=(7.8, 3.9), constrained_layout=True)
for r in rhos_to_plot:
    ax.plot(
        alpha_grid,
        rhs_per_rho[r],
        marker=markers[r],
        markersize=3.0,
        linewidth=1.0,
        color=colors[r],
        alpha=0.9,
        label=rf"Jacobian Size, $\rho={r}$",
    )
ax.plot(
    alpha_grid,
    scaled_lower_bound,
    linewidth=2.0,
    linestyle="-",
    color="#d62728",
    label=rf"Lower Bound  (Var(target))",
)

ax.set_xlabel(r"shape parameter $\alpha$", fontsize=12)
ax.set_ylabel(r"size", fontsize=12)
ax.set_yscale("log")
ax.legend(fontsize=10)

plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "poincare_comparison_pareto_correlated.png", dpi=150)
